# UCS420 Cognitive Computing — Lab Assignment 8
## NLP using Python-II
**Name:** Diksha Mahajan  
**Roll Number:** 1024160059  
**Topics:** Text preprocessing, feature extraction, similarity, sentiment analysis and text generation

Run cells in order in Google Colab. Outputs are calculated from the texts below; no results are fabricated.

In [ ]:
!pip -q install nltk scikit-learn textblob wordcloud tensorflow pandas matplotlib
import re, string, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
for item in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(item, quiet=True)
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from collections import Counter
STOP = set(stopwords.words('english'))
print('Setup complete')

## Q1 — Paragraph preprocessing and frequency distribution

In [ ]:
paragraph = ("Artificial intelligence helps students explore creative ways to solve problems. "
             "Modern language models can summarize articles and explain difficult ideas clearly. "
             "Researchers develop responsible systems that protect privacy and reduce errors. "
             "In classrooms, intelligent tools support practice, feedback, and independent learning. "
             "However, students should verify important claims instead of trusting every answer. "
             "I enjoy studying this technology because it combines mathematics, language, and imagination.")
print('Original paragraph:\n', paragraph)
cleaned = re.sub(r'[^\w\s]', '', paragraph.lower())
words = word_tokenize(cleaned)
sentences = sent_tokenize(paragraph)
split_words = paragraph.split()
nltk_words = word_tokenize(paragraph)
filtered = [w for w in words if w not in STOP]
print('\nLowercase, punctuation removed:\n', cleaned)
print('\nSentences:', len(sentences), sentences)
print('\nCleaned word tokens:', words)
print('\nPython split:', split_words)
print('\nNLTK word_tokenize:', nltk_words)
print('\nWithout stopwords:', filtered)
print('\nWord frequencies:')
print(pd.DataFrame(Counter(filtered).most_common(), columns=['Word', 'Frequency']).to_string(index=False))
print('\nComparison: split() separates only on whitespace, so punctuation stays attached; word_tokenize() usually separates punctuation into its own tokens.')

## Q2 — Regex, stemming and lemmatization

In [ ]:
alphabetic_words = re.findall(r'[A-Za-z]+', paragraph.lower())
meaningful_words = [w for w in alphabetic_words if w not in STOP]
porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()
comparison = pd.DataFrame({'Original': meaningful_words,
                           'Porter Stem': [porter.stem(w) for w in meaningful_words],
                           'WordNet Lemma': [lemmatizer.lemmatize(w) for w in meaningful_words]})
print(comparison.to_string(index=False))
print('\nInterpretation: Stemming quickly strips endings and may create non-dictionary forms; lemmatization generally keeps valid dictionary words. WordNetLemmatizer here uses its default noun POS, so some verbs may remain unchanged.')
print('Example comparison:')
for w in ['studying', 'learning', 'applications']:
    print(f'{w}: stem={porter.stem(w)}, lemma={lemmatizer.lemmatize(w)}')
print('Preference: use stemming for fast search/indexing; prefer lemmatization for readable dictionary forms and interpretation.')


## Q3 — Bag of Words and TF-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
texts = [
    'The new laptop has a fast processor and excellent battery life.',
    'The online course teaches Python programming with practical coding projects.',
    'The smart watch tracks daily exercise, sleep quality, and heart rate.'
]
cv = CountVectorizer(stop_words='english')
bow = cv.fit_transform(texts)
print('Bag of Words:')
print(pd.DataFrame(bow.toarray(), columns=cv.get_feature_names_out(), index=['Text 1','Text 2','Text 3']).to_string())
tv = TfidfVectorizer(stop_words='english')
tfidf = tv.fit_transform(texts)
print('\nTF-IDF scores:')
print(pd.DataFrame(tfidf.toarray(), columns=tv.get_feature_names_out(), index=['Text 1','Text 2','Text 3']).round(3).to_string())
features = tv.get_feature_names_out()
for i, row in enumerate(tfidf.toarray()):
    top = sorted([(features[j], float(row[j])) for j in np.where(row > 0)[0]], key=lambda x: (-x[1], x[0]))[:3]
    print(f'Text {i+1} top 3:', top)
    print('Interpretation: These terms represent this text’s main topic and receive the highest TF-IDF weights within the three-text corpus.')
print('Interpretation: Higher TF-IDF weights highlight words that are prominent in a text but less common across the three texts. Tied scores are ordered alphabetically.')

## Q4 — Jaccard and cosine similarity

In [ ]:
ai_text = """Artificial intelligence learns patterns from large datasets.
AI systems can classify images and understand language.
Machine learning models improve with training data.
Developers evaluate accuracy and fairness in applications.
These tools support automation in modern businesses."""
blockchain_text = """Blockchain stores records in a distributed digital ledger.
Networks verify transactions using agreed rules.
Cryptography helps protect data and transaction history.
Developers build applications for secure record sharing.
These systems support transparency in modern businesses."""
def prep(text):
    return [w for w in re.findall(r'[a-z]+', text.lower()) if w not in STOP]
a, b = prep(ai_text), prep(blockchain_text)
set_a, set_b = set(a), set(b)
jaccard = len(set_a & set_b) / len(set_a | set_b) if set_a | set_b else 0
from sklearn.metrics.pairwise import cosine_similarity
vec = TfidfVectorizer(stop_words='english')
mat = vec.fit_transform([ai_text, blockchain_text])
cosine = cosine_similarity(mat[0], mat[1])[0,0]
print('AI tokens:', a)
print('Blockchain tokens:', b)
print('Shared words:', sorted(set_a & set_b))
print(f'Jaccard similarity: {jaccard:.4f}')
print(f'TF-IDF cosine similarity: {cosine:.4f}')
print('Analysis: Jaccard reveals exact vocabulary overlap, while TF-IDF cosine accounts for relative word importance. For these short technology descriptions, cosine can be more informative when shared terms vary in importance; neither captures synonyms reliably.')

## Q5 — Sentiment, polarity, subjectivity and positive-review word cloud

In [ ]:
from textblob import TextBlob
from wordcloud import WordCloud
reviews = [
    'This laptop is excellent, reliable, and surprisingly fast.',
    'I love the clear display and the wonderful battery life.',
    'The keyboard is uncomfortable and the service is disappointing.',
    'The package arrived yesterday and includes a charger.',
    'The app is useful and easy to navigate.'
]
records = []
for review in reviews:
    sentiment = TextBlob(review).sentiment
    label = 'Positive' if sentiment.polarity > 0 else ('Negative' if sentiment.polarity < 0 else 'Neutral')
    records.append({'Review': review, 'Polarity': round(sentiment.polarity, 3), 'Subjectivity': round(sentiment.subjectivity, 3), 'Class': label})
df_reviews = pd.DataFrame(records)
print(df_reviews.to_string(index=False))
positive_text = ' '.join(df_reviews.loc[df_reviews['Class'] == 'Positive', 'Review'])
if positive_text.strip():
    cloud = WordCloud(width=900, height=450, background_color='white', stopwords=STOP, random_state=42).generate(positive_text)
    plt.figure(figsize=(11,5)); plt.imshow(cloud, interpolation='bilinear'); plt.axis('off'); plt.title('Positive Reviews Word Cloud'); plt.show()
else:
    print('No positive reviews for word cloud.')
print('Polarity ranges from -1 to +1; subjectivity from 0 (objective) to 1 (subjective). Lexicon-based predictions can miss context and sarcasm.')

## Q6 — Train a simple neural text generator

In [ ]:
training_text = ("Technology changes the way students learn and share ideas. "
"Creative students use computers to design helpful applications. "
"A small project begins with a clear question and a simple plan. "
"Students collect useful examples and prepare clean data for learning. "
"A model discovers patterns in the data and makes predictions. "
"Careful testing helps students find mistakes and improve results. "
"Good developers explain their methods and respect user privacy. "
"Teamwork encourages fresh ideas and thoughtful solutions. "
"Learning Python makes it easier to explore language and artificial intelligence. "
"Practice and curiosity help students build reliable technology for everyday life. "
"Every new experiment teaches a valuable lesson about problem solving. "
"Responsible innovation combines imagination with careful evaluation.")
print('Training word count:', len(training_text.split()))
# TensorFlow versions used by Colab provide Tokenizer via this legacy-compatible import.
try:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
except ImportError:
    from keras.src.legacy.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalAveragePooling1D
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
tf.keras.utils.set_random_seed(42)
tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts([training_text])
encoded = tokenizer.texts_to_sequences([training_text])[0]
sequences = [encoded[:i+1] for i in range(1, len(encoded))]
max_len = max(len(seq) for seq in sequences)
padded = pad_sequences(sequences, maxlen=max_len, padding='pre')
X, y = padded[:, :-1], padded[:, -1]
vocab_size = len(tokenizer.word_index) + 1
model = Sequential([Embedding(vocab_size, 24), GlobalAveragePooling1D(), Dense(48, activation='relu'), Dense(vocab_size, activation='softmax')])
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print('Sequences:', len(sequences), 'Vocabulary:', vocab_size)
history = model.fit(X, y, epochs=35, batch_size=16, verbose=0)
print('Final training accuracy:', round(float(history.history['accuracy'][-1]), 3))
reverse_words = {idx: word for word, idx in tokenizer.word_index.items()}
def generate_line(seed, next_words=12):
    text = seed
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([text])[0]
        seq = pad_sequences([seq], maxlen=max_len-1, padding='pre')
        probs = model.predict(seq, verbose=0)[0]
        probs[0] = 0  # ignore padding token
        predicted = int(np.argmax(probs))
        word = reverse_words.get(predicted)
        if not word: break
        text += ' ' + word
    return text
print('\nGenerated lines (results depend on training):')
for seed in ['students', 'technology', 'learning']:
    print('-', generate_line(seed))
print('Note: This is a small demonstration corpus; generated lines may repeat words or be ungrammatical. A larger dataset and more training would be needed for fluent output.')

## Submission checklist
- Run **Runtime → Run all** in Colab and inspect outputs and plots.
- Confirm Q1–Q6 execute without errors.
- Save/download the completed `.ipynb` file with your roll number.
- The Q6 output is generated by the trained model at runtime, not prewritten.